In [1]:
!pwd

/home/hgkahng/Workspaces/soft-prompt/notebooks/imdb


In [2]:
import os
import sys
sys.path.insert(
    0, os.path.abspath("../../")  # two levels up
)
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegressionCV

In [3]:
root_dir = Path("../../").resolve()
data_dir = root_dir / "data/imdb"
print(*os.listdir(data_dir), sep="\n")

test.csv
train.csv
embeddings
.DS_Store


Load text

In [4]:
# load data
train_data = pd.read_csv(data_dir / "train.csv")
test_data = pd.read_csv(data_dir / "test.csv")

In [5]:
train_data.head()

,sentiment,review
0,positive,for a movie that gets no respect there sure ar...
1,positive,bizarre horror movie filled with famous faces ...
2,positive,"a solid, if unremarkable film. matthau, as ein..."
3,positive,it is a strange feeling to sit alone in a thea...
4,positive,"you probably all already know this by now, but..."


In [6]:
test_data.head()

,sentiment,review
0,positive,"based on an actual story, john boorman shows t..."
1,positive,this is a gem. as a film four production the a...
2,positive,"i really like this show. it has drama, romance..."
3,positive,this is the best d experience disney has at th...
4,positive,"of the korean movies i have seen, only three h..."


Load embeddings

In [8]:
emb_save_dir = data_dir / "embeddings/openai/text-embedding-3-small"
print(*os.listdir(emb_save_dir), sep="\n")

train.features.npy
train.labels.npy
test.features.npy
test.labels.npy


In [9]:
# load embeddings & labels
X_train = np.load(emb_save_dir / "train.features.npy")
y_train = np.load(emb_save_dir / "train.labels.npy")
X_test  = np.load(emb_save_dir / "test.features.npy")
y_test  = np.load(emb_save_dir / "test.labels.npy")

## Classification Performance

In [10]:
from sklearn.linear_model import LogisticRegressionCV

def evaluate_lg(X_train: np.ndarray,
                y_train: np.ndarray,
                X_test: np.ndarray,
                y_test: np.ndarray,
                subsample_size: int = 1000,
                bootstrap: bool = True,  # sampling with replacement
                n_trials: int = 50) -> dict[str, tuple[float, float]]:

    assert X_train.shape[0] == len(y_train)

    train_acc_array = np.empty(n_trials)
    test_acc_array = np.empty_like(train_acc_array)
    
    original_idx = np.arange(X_train.shape[0])  # [0, 1, ..., len(X_train)]

    for i in range(n_trials):
        
        # get indices to use for training
        rng = np.random.default_rng(42+i)
        if bootstrap:
            use_idx = rng.choice(original_idx, size=subsample_size, replace=True)
        else:
            shuffled_idx = rng.permutation(original_idx)
            use_idx = shuffled_idx[:subsample_size]

        # fit model
        lg = LogisticRegressionCV(Cs=10, cv=5, penalty='l2',
                                  solver='lbfgs', max_iter=1000, n_jobs=8,
                                  random_state=42+i)
        lg.fit(X_train[use_idx], y_train[use_idx]);

        # evaluate
        train_acc_array[i] = lg.score(X_train[use_idx], y_train[use_idx])
        test_acc_array[i] = lg.score(X_test, y_test)

    return {
        'train_accuracy': (train_acc_array.mean(), train_acc_array.std(ddof=1)),
        'test_accuracy': (test_acc_array.mean(), test_acc_array.std(ddof=1)),
    }

In [11]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [12]:
size_to_acc = {}

N = len(X_train)
ratios = (0.01, 0.05, 0.10, 0.25, 0.50, 1.00)
subsample_sizes = [int(N * r) for r in ratios]
for i, subsample_size in enumerate(subsample_sizes):

    print(f">> Sample size: {subsample_size:>6,} (r={ratios[i]:.2f})")

    eval_result = evaluate_lg(
        X_train,
        (y_train > 0.5).astype(int),
        X_test,
        (y_test > 0.5).astype(int),
        subsample_size=subsample_size,
        bootstrap=True,
        n_trials=50,
    )

    train_acc, test_acc = \
        eval_result['train_accuracy'], eval_result['test_accuracy']

    size_to_acc[subsample_size] = eval_result

    print("\t Train acc = {:.4f} ({:.4f})".format(*train_acc))
    print("\t  Test acc = {:.4f} ({:.4f})".format(*test_acc))

>> Sample size:    250 (r=0.01)
	 Train acc = 0.9719 (0.0249)
	  Test acc = 0.9159 (0.0070)
>> Sample size:  1,250 (r=0.05)
	 Train acc = 0.9660 (0.0173)
	  Test acc = 0.9309 (0.0035)
>> Sample size:  2,500 (r=0.10)
	 Train acc = 0.9612 (0.0133)
	  Test acc = 0.9349 (0.0029)
>> Sample size:  6,250 (r=0.25)
	 Train acc = 0.9655 (0.0105)
	  Test acc = 0.9371 (0.0032)
>> Sample size: 12,500 (r=0.50)
	 Train acc = 0.9665 (0.0039)
	  Test acc = 0.9381 (0.0022)
>> Sample size: 25,000 (r=1.00)
	 Train acc = 0.9554 (0.0055)
	  Test acc = 0.9414 (0.0036)


## Diversity Metrics

In [ ]:
from softprompt.metrics.diversity import (
    vocabulary_size,
    distinct_n,
    average_pairwise_similarity,
    average_pairwise_similarity_by_class,
    inter_sample_ngram_freq,
)

`Vocabulary Size`

In [11]:
%%time

vocab_size_tr = vocabulary_size(train_data['review'].tolist())
vocab_size_te = vocabulary_size(test_data['review'].tolist())
print(f"[Train] Vocab: {vocab_size_tr:>7,}")
print(f" [Test] Vocab: {vocab_size_te:>7,}")

[Train] Vocab:  77,169
 [Test] Vocab:  75,974
CPU times: user 35.7 s, sys: 63.6 ms, total: 35.8 s
Wall time: 36 s


`Distinct-n`

In [11]:
%%time

distinct_2_tr = distinct_n(train_data['review'].tolist(), n=2)
distinct_2_te = distinct_n(test_data['review'].tolist(), n=2)
print(f"[Train] Distinct-2: {distinct_2_tr:.4f}")
print(f" [Test] Distinct-2: {distinct_2_te:.4f}")

[Train] Distinct-2: 0.1912
 [Test] Distinct-2: 0.1915
CPU times: user 35.8 s, sys: 186 ms, total: 36 s
Wall time: 36.3 s


`Average Pairwise Similarity`

In [14]:
%%time

avg_ps_tr = average_pairwise_similarity(X_train)
print(f"[Train] Avg Pairwise Similarity: {avg_ps_tr:.4f}")

avg_ps_te = average_pairwise_similarity(X_test)
print(f" [Test] Avg Pairwise Similarity: {avg_ps_te:.4f}")

[Train] Avg Pairwise Similarity: 0.3753
 [Test] Avg Pairwise Similarity: 0.3745
CPU times: user 1min 4s, sys: 6.46 s, total: 1min 11s
Wall time: 16.8 s


`Inter/Intra-Class Pairwise Similarity`

In [ ]:
inter_sim_tr, intra_sim_tr = average_pairwise_similarity_by_class(X_train, y_train)
print(f"[Train] Inter-class APS: {inter_sim_tr:.4f}, Intra-class APS: {intra_sim_tr:.4f}")

inter_sim_te, intra_sim_te = average_pairwise_similarity_by_class(X_test, y_test)
print(f" [Test] Inter-class APS: {inter_sim_te:.4f}, Intra-class APS: {intra_sim_te:.4f}")

[Train] Intra-class APS: 0.3950, Inter-class APS: 0.3555
 [Test] Intra-class APS: 0.3944, Inter-class APS: 0.3545


In [30]:
%%time

self_bleu = compute_self_bleu_corpus(
    train_data['review'].tolist(), reference_size=100
)
print(f"Self-BLEU: {self_bleu:.4f}")

Computing self-BLEU...: 0it [00:00, ?it/s]

/opt/miniconda3/envs/langchain/lib/python3.11/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)


Self-BLEU: 0.1526
CPU times: user 12min 56s, sys: 5.91 s, total: 13min 2s
Wall time: 13min 7s


`Average Pairwise Distance`

In [23]:
avg_dist = compute_avg_pairwise_distance(X_train)
print(f"Average Pairwise Embedding Distance: {avg_dist:.4f}")

Average Pairwise Embedding Distance: 1.1147
